# Budget vs Actual Variance Analysis

## Project Overview
Analyze budget and actual spending across departments and periods to identify meaningful variances, their likely explanations, and patterns of over/underspending. This is a core financial planning and analysis (FP&A) exercise.

## Learning Objectives
- Calculate budget-to-actual variances (absolute and percentage)
- Identify favorable and unfavorable variances across departments and time
- Detect systematic over- or under-budgeting patterns
- Generate actionable variance explanations and recommendations

## Problem Statement
A company's CFO wants a quarterly variance report that highlights which departments are consistently over or under budget, the magnitude of variances, and whether the budgeting process needs recalibration.

## Why This Project Matters
Budget vs actual analysis is the most fundamental FP&A activity. It holds departments accountable, improves forecasting accuracy over time, and surfaces operational issues that might otherwise go unnoticed.

## Dataset Overview
Synthetic budget and actual data: 6 departments × 8 quarters (2 years) with line-item categories. Includes realistic patterns: some departments consistently overspend, others underspend, and seasonal effects create predictable variances.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by typical corporate FP&A reporting
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
VARIANCE_THRESHOLD = 0.10  # flag variances > 10%
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)

departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Operations', 'Finance']
categories = ['Personnel', 'Software', 'Travel', 'Marketing Spend', 'Equipment', 'Consulting', 'Other']
quarters = pd.period_range('2023Q1', '2024Q4', freq='Q')

# Budget allocation per dept per category per quarter (base, in $K)
dept_base = {'Engineering': 500, 'Marketing': 400, 'Sales': 350,
             'HR': 200, 'Operations': 300, 'Finance': 180}
cat_split = {'Personnel': 0.40, 'Software': 0.15, 'Travel': 0.08, 'Marketing Spend': 0.12,
             'Equipment': 0.08, 'Consulting': 0.10, 'Other': 0.07}

# Variance patterns (systematic bias per dept)
dept_bias = {'Engineering': 1.08, 'Marketing': 1.15, 'Sales': 0.92,
             'HR': 1.02, 'Operations': 1.05, 'Finance': 0.95}

rows = []
for q in quarters:
    q_idx = quarters.get_loc(q)
    for dept in departments:
        for cat in categories:
            budget = dept_base[dept] * cat_split[cat] * 1000  # to dollars

            # Seasonal adjustments
            if cat == 'Travel' and q.quarter in [2, 3]:
                budget *= 1.3
            if cat == 'Marketing Spend' and q.quarter == 4:
                budget *= 1.5

            # Actual = budget * bias + noise
            actual = budget * dept_bias[dept] * np.random.normal(1.0, 0.08)

            # Special events
            if dept == 'Marketing' and cat == 'Marketing Spend' and q == pd.Period('2024Q4'):
                actual *= 1.4  # big campaign overshoot
            if dept == 'Sales' and cat == 'Travel' and q.quarter == 1:
                actual *= 0.6  # sales kickoff cancelled

            budget = round(max(budget, 1000), 2)
            actual = round(max(actual, 500), 2)

            rows.append({
                'Quarter': str(q), 'Department': dept, 'Category': cat,
                'Budget': budget, 'Actual': actual,
            })

df = pd.DataFrame(rows)
df['Variance'] = df['Actual'] - df['Budget']
df['VariancePct'] = ((df['Actual'] - df['Budget']) / df['Budget'] * 100).round(2)
df['Status'] = df['VariancePct'].apply(lambda x: 'Over Budget' if x > 5 else 'Under Budget' if x < -5 else 'On Track')

print(f'Dataset shape: {df.shape}')
print(f'Total budget: ${df["Budget"].sum():,.2f}')
print(f'Total actual: ${df["Actual"].sum():,.2f}')
print(f'Net variance: ${df["Variance"].sum():,.2f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Quarters: {df["Quarter"].nunique()}')
print(f'Departments: {df["Department"].nunique()}')
print(f'Categories: {df["Category"].nunique()}')
print(f'Records: {len(df)}')
print(f'\nVariance status:\n{df["Status"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall budget vs actual by department
dept_summary = df.groupby('Department')[['Budget', 'Actual']].sum()
dept_summary.plot.bar(ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Budget vs Actual by Department')
axes[0,0].set_ylabel('Amount ($)')
axes[0,0].tick_params(axis='x', rotation=45)

# Variance % by department
dept_var_pct = ((dept_summary['Actual'] - dept_summary['Budget']) / dept_summary['Budget'] * 100).sort_values()
colors = ['#66bb6a' if v < 0 else '#ef5350' for v in dept_var_pct]
dept_var_pct.plot.barh(ax=axes[0,1], color=colors, edgecolor='black')
axes[0,1].set_title('Variance % by Department')
axes[0,1].set_xlabel('Variance %')
axes[0,1].axvline(x=0, color='grey', linestyle='--')

# Quarterly total variance
q_summary = df.groupby('Quarter')[['Budget', 'Actual']].sum()
q_summary['Variance'] = q_summary['Actual'] - q_summary['Budget']
q_summary['Variance'].plot.bar(ax=axes[1,0], edgecolor='black',
    color=['#ef5350' if v > 0 else '#66bb6a' for v in q_summary['Variance']])
axes[1,0].set_title('Net Variance by Quarter')
axes[1,0].set_ylabel('Variance ($)')
axes[1,0].axhline(y=0, color='grey', linestyle='--')
axes[1,0].tick_params(axis='x', rotation=45)

# Status distribution
df['Status'].value_counts().plot.pie(ax=axes[1,1], autopct='%1.0f%%', colors=['#ef5350','#66bb6a','#ffa726'])
axes[1,1].set_title('Variance Status Distribution')
axes[1,1].set_ylabel('')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Variance Deep Dive

In [ ]:
dept_detail = df.groupby('Department').agg(
    total_budget=('Budget', 'sum'),
    total_actual=('Actual', 'sum'),
    total_variance=('Variance', 'sum'),
    avg_variance_pct=('VariancePct', 'mean'),
    count_over=('Status', lambda x: (x == 'Over Budget').sum()),
    count_under=('Status', lambda x: (x == 'Under Budget').sum()),
).round(2)
dept_detail['overall_variance_pct'] = ((dept_detail['total_actual'] - dept_detail['total_budget']) / dept_detail['total_budget'] * 100).round(2)
dept_detail = dept_detail.sort_values('overall_variance_pct', ascending=False)
print('Department Variance Summary:')
print(dept_detail.to_string())

## Category Variance Heatmap

In [ ]:
cat_dept_var = df.groupby(['Department', 'Category'])['VariancePct'].mean().unstack(fill_value=0)
plt.figure(figsize=(12, 6))
sns.heatmap(cat_dept_var, annot=True, fmt='.1f', cmap='RdYlGn_r', center=0)
plt.title('Avg Variance % by Department × Category (Red = Over Budget)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'variance_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Quarterly Trend by Department

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, dept in zip(axes.flat, departments):
    dept_q = df[df['Department'] == dept].groupby('Quarter')[['Budget', 'Actual']].sum()
    dept_q.plot(ax=ax, marker='o')
    ax.set_title(f'{dept}')
    ax.set_ylabel('$')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Budget vs Actual Trend by Department', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quarterly_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Largest Individual Variances

In [ ]:
flagged = df[abs(df['VariancePct']) > VARIANCE_THRESHOLD * 100].sort_values('Variance', key=abs, ascending=False)
print(f'Line items exceeding {VARIANCE_THRESHOLD:.0%} variance: {len(flagged)} ({len(flagged)/len(df):.0%})')
print(f'\nTop 15 largest variances:')
print(flagged[['Quarter','Department','Category','Budget','Actual','Variance','VariancePct']].head(15).to_string(index=False))

## Budgeting Accuracy Score

In [ ]:
# MAPE per department (how accurate is their budgeting?)
dept_mape = df.groupby('Department').apply(
    lambda x: (abs(x['Actual'] - x['Budget']) / x['Budget']).mean() * 100
).round(2).sort_values()
print('Budget Accuracy (MAPE — lower is better):')
for dept, mape in dept_mape.items():
    print(f'  {dept}: {mape:.1f}% average error')

fig, ax = plt.subplots(figsize=(8, 5))
dept_mape.plot.bar(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Budget Accuracy by Department (MAPE)')
ax.set_ylabel('Mean Absolute Percentage Error (%)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'budget_accuracy.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Marketing** consistently overspends (~15% average), driven primarily by Marketing Spend and Consulting categories — budgets may need upward revision or spending controls
- **Sales** consistently underspends (~8% below budget), particularly on Travel — may indicate conservative budgeting or cancelled activities
- **Engineering** shows moderate systematic overspend (~8%), largely in Software and Personnel
- **Q4** variances are largest due to year-end campaigns and budget flush spending
- **Finance** and **HR** are closest to budget, suggesting more predictable cost structures
- The budgeting process should be recalibrated: departments with systematic bias need adjusted baseline budgets

## Limitations
- Synthetic data with pre-programmed biases — real variances are noisier
- No revenue-side variance (only expense analysis)
- No multi-year trend to assess whether budgeting accuracy is improving
- No line-item detail below category level
- No re-forecasting or rolling forecast comparison

## How to Improve This Project
- Use real ERP budget and actual data for more realistic analysis
- Add revenue variance analysis for a complete P&L view
- Include rolling forecasts and forecast-vs-actual comparison
- Add YoY variance comparison for trend analysis
- Build automated variance commentary generation

## Production Considerations
- Automate monthly/quarterly variance reports with email distribution
- Set up escalation rules: variances > 15% trigger department head review
- Track budget accuracy improvement over time as a KPI
- Integrate with ERP for real-time budget tracking

## Common Mistakes
- Treating all variances equally (a 20% variance on a $10K line item matters less than 5% on $1M)
- Not distinguishing between timing variances (spend shifted to next quarter) and permanent variances
- Blaming departments for variances caused by external factors (price increases, FX changes)
- Using last year's budget as next year's without adjusting for known changes

## Mini Challenge / Exercises
1. Which department-category combination has the single largest cumulative variance over 2 years?
2. Calculate the "rolling budget accuracy" quarter by quarter for each department. Is it improving?
3. If budgets were set at 105% of historical actuals, how many variances would still exceed 10%?
4. Rank departments by consistency (lowest standard deviation of variance %).

## Final Summary / Key Takeaways
- Budget vs actual analysis is the cornerstone of financial accountability
- Systematic over/underspending signals budgeting process issues, not just spending issues
- Seasonal effects should be built into budgets to reduce predictable variances
- Variance analysis should always include both magnitude and materiality
- The goal is not zero variance but improving forecast accuracy over time